In [1]:
import pandas as pd

In [ ]:
import fitz

pdf_path = '../data/Pratik_Resume-3.pdf'

doc = fitz.open(pdf_path)
text = ""

for page in doc:
    text += page.get_text()

print(text[:2000])

In [ ]:
print("Number of characters:", len(text))
print("\nFirst 3000 characters:\n")
print(text[:3000])

In [ ]:
print(text[:1000])

In [ ]:
print(text[1000:2000])

In [ ]:
print(text[2000:4000])

In [7]:
import re

def split_resume_sections(text):
    sections = {}

    section_names = [
        'education',
        'skills',
        'objective',
        'experience',
        'projects',
        'summary',
        'profile'
    ]

    pattern = r"(?im)^\s*(" + "|".join(section_names) + r")\s*$"
    matches = list(re.finditer(pattern,text))

    for i,match in enumerate(matches):
        section_name = match.group(1).lower()

        start = match.end()
        if i+1 < len(matches):
            end = matches[i+1].start()
        else:
            end = len(text)

        section_text = text[start:end].strip()
        sections[section_name] = section_text

    return sections

In [ ]:
sections = split_resume_sections(text)

print(sections.keys())

In [ ]:
for section,content in sections.items():
    print(section.upper())
    print('=' * 60)
    print(content[:1000])

In [10]:
candidate_profile = {
    'name':None,
    'email':None,
    'linkedin':None,
    'github':None,
    'phone':None,
    'education':[],
    'skills':[],
    'experience':[],
    'projects':[]
}

In [11]:
def extract_contact_information(text):
    profile = {}

    email_match = re.search(
        r'[\w\.-]+@[\w\.-]+\.\w+',
        text
    )
    profile['email'] = (
        email_match.group(0)
        if email_match
        else None
    )

    phone_match = re.search(
        r'(?<!\d)(?:\+91[\s-]?)?[6-9]\d{9}(?!\d)',
        text
    )
    profile['phone'] = (
        phone_match.group(0)
        if phone_match
        else None
    )

    linkedin_match = re.search(
        r'LinkedIn:\s*([^\n|]+)',
        text,
        re.IGNORECASE
    )
    profile["linkedin"] = (
        linkedin_match.group(1).strip()
        if linkedin_match
        else None
    )

    github_match = re.search(
        r'Github:\s*([^\n|]+)',
        text,
        re.IGNORECASE
    )
    profile["github"] = (
        github_match.group(1).strip()
        if github_match
        else None
    )

    return profile

In [ ]:
contact_info = extract_contact_information(text)
contact_info

In [13]:
def extract_name(text):
    first_line = text.strip().split('\n')[0]
    return first_line.strip()

In [ ]:
name = extract_name(text)
print(name)

In [ ]:
candidate_profile = {
    "name": name,
    "email": contact_info["email"],
    "phone": contact_info["phone"],
    "linkedin": contact_info["linkedin"],
    "github": contact_info["github"],
    "skills": sections.get("skills"),
    "education": sections.get("education"),
    "experience": sections.get("experience"),    
    "projects": sections.get("projects"),
}
print(candidate_profile)

In [ ]:
from pathlib import Path

candidate_profile["linkedin_url"] = ""
candidate_profile["github_url"] = ""
candidate_profile["portfolio_url"] = ""
candidate_profile["twitter_url"] = ""
candidate_profile["other_url"] = ""

candidate_profile["location"] = "Patna"
candidate_profile["org"] = ""

# Resume PDF
candidate_profile["resume_path"] = str(
    (Path.cwd().parent / "data" / "Pratik_Resume-3.pdf").resolve()
)

print("Resume path:")
print(candidate_profile["resume_path"])

In [ ]:
from pathlib import Path

resume_path = Path(candidate_profile["resume_path"])

print("Exists:", resume_path.exists())
print("Path:", resume_path)

In [16]:
def clean_section_text(text):
    if not text:
        return None

    # Replace unusual whitespace characters
    text = text.replace("\u200b", "")
    text = text.replace("\xa0", " ")

    # Normalize whitespace
    text = re.sub(r"[ \t]+", " ", text)

    # Normalize excessive newlines
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()

In [17]:
for key in ["skills", "education", "experience", "projects"]:
    if candidate_profile.get(key):
        candidate_profile[key] = clean_section_text(
            candidate_profile[key]
        )

In [ ]:
print(candidate_profile["skills"])
print(candidate_profile["experience"])

In [19]:
def extract_skills(skills_text):
    if not skills_text:
        return []

    skills = []

    for line in skills_text.split("\n"):
        line = line.strip()

        if not line:
            continue

        # Remove category before colon
        if ":" in line:
            _, skill_text = line.split(":", 1)
        else:
            skill_text = line

        # Split individual skills
        parts = skill_text.split(",")

        for skill in parts:
            skill = skill.strip()

            if skill:
                skills.append(skill)

    # Remove duplicates while preserving order
    skills = list(dict.fromkeys(skills))

    return skills

In [20]:
candidate_profile["skills"] = extract_skills(
    candidate_profile["skills"]
)

In [ ]:
print(candidate_profile["skills"])

In [22]:
def clean_section_text(text):
    if not text:
        return None

    # Replace unusual whitespace characters
    text = text.replace("\u200b", "")
    text = text.replace("\xa0", " ")

    # Normalize whitespace
    text = re.sub(r"[ \t]+", " ", text)

    # Normalize excessive newlines
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()

In [23]:
for key in ["education", "experience", "projects"]:
    if candidate_profile.get(key):
        candidate_profile[key] = clean_section_text(
            candidate_profile[key]
        )

In [24]:
candidate_profile["experience_text"] = candidate_profile["experience"]
candidate_profile["projects_text"] = candidate_profile["projects"]

del candidate_profile["experience"]
del candidate_profile["projects"]

In [ ]:
print("Skills:")
print(candidate_profile["skills"])

print("\nEducation:")
print(candidate_profile["education"])

print("\nExperience:")
print(candidate_profile["experience_text"][:500])

print("\nProjects:")
print(candidate_profile["projects_text"][:500])

In [26]:
def build_candidate_text(profile):
    sections = []

    if profile.get("skills"):
        skills = ", ".join(profile["skills"])
        sections.append(f"Skills: {skills}")

    if profile.get("education"):
        sections.append(f"Education: {profile['education']}")

    if profile.get("experience_text"):
        sections.append(
            f"Experience: {profile['experience_text']}"
        )

    if profile.get("projects_text"):
        sections.append(
            f"Projects: {profile['projects_text']}"
        )

    return "\n\n".join(sections)

In [ ]:
candidate_text = build_candidate_text(candidate_profile)

print(candidate_text[:5000])

In [ ]:
from sentence_transformers import SentenceTransformer
embedding_model = SentenceTransformer(
    "BAAI/bge-small-en-v1.5"
)

In [29]:
candidate_embedding = embedding_model.encode(
    candidate_text,
    normalize_embeddings = True
)

In [30]:
print(type(candidate_embedding))
print(candidate_embedding.shape)

<class 'numpy.ndarray'>
(384,)


In [31]:
job_1 = """
We are looking for a Data Analyst with strong Python and SQL skills.
The candidate will work with large datasets, build dashboards,
perform data analysis, and generate business insights.
"""

job_2 = """
We are looking for a Machine Learning Engineer experienced in Python,
machine learning, deep learning, model development, and deployment.
Experience with FastAPI and Docker is preferred.
"""

job_3 = """
We are looking for a Civil Engineer responsible for structural design,
construction planning, site supervision, and infrastructure projects.
"""

In [32]:
job_embeddings = embedding_model.encode(
    [job_1,job_2,job_3],
    normalize_embeddings=True
)

In [33]:
from sklearn.metrics.pairwise import cosine_similarity

similarites = cosine_similarity(
    [candidate_embedding],
    job_embeddings
)[0]

print('Data Analyst',similarites[0])
print('ML Engineer',similarites[1])
print('Civil Engineer',similarites[2])

Data Analyst 0.75222766
ML Engineer 0.76550925
Civil Engineer 0.5181116


In [34]:
import pandas as pd

# Login using e.g. `huggingface-cli login` to access this dataset
splits = {'train': 'train.csv', 'validation': 'validation.csv'}
df = pd.read_csv("hf://datasets/0xnbk/resume-ats-score-v1-en/" + splits["train"])

In [35]:
df

,text,ats_score,original_label
0,SummaryI am seeking a position wherein I may u...,80.6,Good Fit
1,ProfileHighly motivated Sales Associate with e...,24.3,No Fit
2,SummaryHaving achieved a milestone of 7 years ...,53.9,Potential Fit
3,SummaryWireless communications engineer with e...,52.5,Potential Fit
4,SummaryData Entry experienced and adept at inp...,59.2,Potential Fit
...,...,...,...
5094,Experience10 2019to06 2019Data AnalystAngi Hom...,25.8,No Fit
5095,SummaryHaving achieved a milestone of 7 years ...,56.1,Potential Fit
5096,Professional SummaryRELATED KNOWLEDGE AND SKIL...,27.4,No Fit
5097,SummaryObtain a position where I can utilize m...,21.7,No Fit


In [36]:
sample = df.iloc[0]

print("ATS SCORE:", sample["ats_score"])
print("LABEL:", sample["original_label"])
print("\nFULL TEXT:\n")
print(sample["text"])

ATS SCORE: 80.6
LABEL: Good Fit

FULL TEXT:

SummaryI am seeking a position wherein I may use my proven development, technical, and analytical skills while having the opportunity to continue my learning and diversify my experiences. Highlights17 years experience in Information TechnologyHeld a Secret security clearanceExpert in business process including requirements gathering, analysis, design, development, testing and implementation of software applicationsStrong understanding of structured programming techniques and Software Development life Cycle (SDLC) - Agile (SCRUM)Able to learn, understand, and apply new software programs easily.Interface design and implementationAgile development methodologiesProject management and Requirements analysis and design phasesProcess improvementResults-orientedStaff developmentExcellent communication, analytical, interpersonal, and presentation skillsExpert ability at managing multiple projects simultaneously Experience06 2011to11 2015Manager of Sof

In [37]:
print(repr(df.iloc[0]["text"][:300]))

'SummaryI am seeking a position wherein I may use my proven development, technical, and analytical skills while having the opportunity to continue my learning and diversify my experiences. Highlights17 years experience in Information TechnologyHeld a Secret security clearanceExpert in business proces'


In [38]:
separator = " SEP "

print(
    "Rows containing separator:",
    df["text"].str.contains(separator, regex=False).sum()
)

print(
    "Total rows:",
    len(df)
)

Rows containing separator: 5099
Total rows: 5099


In [39]:
df[["resume_text", "job_text"]] = df["text"].str.split(
    " SEP ",
    n=1,
    expand=True
)

In [40]:
print("Resume:")
print(df["resume_text"].iloc[0][:1000])

print("\nJob:")
print(df["job_text"].iloc[0][:1000])

Resume:
SummaryI am seeking a position wherein I may use my proven development, technical, and analytical skills while having the opportunity to continue my learning and diversify my experiences. Highlights17 years experience in Information TechnologyHeld a Secret security clearanceExpert in business process including requirements gathering, analysis, design, development, testing and implementation of software applicationsStrong understanding of structured programming techniques and Software Development life Cycle (SDLC) - Agile (SCRUM)Able to learn, understand, and apply new software programs easily.Interface design and implementationAgile development methodologiesProject management and Requirements analysis and design phasesProcess improvementResults-orientedStaff developmentExcellent communication, analytical, interpersonal, and presentation skillsExpert ability at managing multiple projects simultaneously Experience06 2011to11 2015Manager of Software DevelopmentAvidxchange Birmingh

In [41]:
print("Missing resume text:", df["resume_text"].isna().sum())
print("Missing job text:", df["job_text"].isna().sum())

Missing resume text: 0
Missing job text: 0


In [42]:
df.head()

,text,ats_score,original_label,resume_text,job_text
0,SummaryI am seeking a position wherein I may u...,80.6,Good Fit,SummaryI am seeking a position wherein I may u...,- Share resume to shan imrsoft.com Job Title ...
1,ProfileHighly motivated Sales Associate with e...,24.3,No Fit,ProfileHighly motivated Sales Associate with e...,Purpose StatementThe Software Engineering Man...
2,SummaryHaving achieved a milestone of 7 years ...,53.9,Potential Fit,SummaryHaving achieved a milestone of 7 years ...,"Title Business AnalystLocation Santa Clara, C..."
3,SummaryWireless communications engineer with e...,52.5,Potential Fit,SummaryWireless communications engineer with e...,Job Description As a Principal Software Engin...
4,SummaryData Entry experienced and adept at inp...,59.2,Potential Fit,SummaryData Entry experienced and adept at inp...,"Title Data EngineerLocation fully remote, how..."


In [43]:
val_df = pd.read_csv(
    "hf://datasets/0xnbk/resume-ats-score-v1-en/validation.csv"
)

print("Training:", df.shape)
print("Validation:", val_df.shape)

Training: (5099, 5)
Validation: (1275, 3)


In [44]:
separator = " SEP "

val_df[["resume_text", "job_text"]] = val_df["text"].str.split(
    separator,
    n=1,
    expand=True
)

(val_df[[
    "resume_text",
    "job_text",
    "ats_score",
    "original_label"
]].head())

,resume_text,job_text,ats_score,original_label
0,SummaryFinancial Accountant specializing in fi...,Role - Business Analyst - Mobile Location - L...,27.6,No Fit
1,SummaryExperienced Data Analyst who responds t...,( Candidates without any Visa Constraints) ( ...,57.6,Potential Fit
2,SummaryDigital marketing professional experien...,Brief Description Who is Monarch Tractor? At ...,22.4,No Fit
3,Professional SummaryHighly motivated Sales Ass...,About Hallgate Management Hallgate Management...,28.0,No Fit
4,Professional SummaryCurrently working with Cat...,"Pay Range 81k - 102k, depending on experience...",58.3,Potential Fit


In [45]:
print("Missing resume:", val_df["resume_text"].isna().sum())
print("Missing job:", val_df["job_text"].isna().sum())

Missing resume: 0
Missing job: 0


In [46]:
df["score"] = df["ats_score"] / 100
val_df["score"] = val_df["ats_score"] / 100

In [47]:
print(df["score"].min(), df["score"].max())

0.1916 0.9005


In [48]:
from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator
baseline_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

/var/folders/x2/vkkqdf2j73qc3q41102n2jt80000gn/T/ipykernel_33325/1712001032.py:1: DeprecationWarning: Importing from 'sentence_transformers.evaluation' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.evaluation' instead.
  from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator


In [49]:
baseline_evaluator = EmbeddingSimilarityEvaluator(
    sentences1=val_df["resume_text"].tolist(),
    sentences2=val_df["job_text"].tolist(),
    scores=val_df["score"].tolist(),
    main_similarity="cosine",
    name="baseline"
)

In [50]:
baseline_results = baseline_evaluator(baseline_model)

print(baseline_results)

{'baseline_pearson_cosine': 0.2140447009238954, 'baseline_spearman_cosine': 0.20930305950683958}


In [51]:
from datasets import Dataset

train_dataset = Dataset.from_dict({
    "sentence1": df["resume_text"].tolist(),
    "sentence2": df["job_text"].tolist(),
    "score": df["score"].tolist()
})

val_dataset = Dataset.from_dict({
    "sentence1": val_df["resume_text"].tolist(),
    "sentence2": val_df["job_text"].tolist(),
    "score": val_df["score"].tolist()
})

In [52]:
model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

In [53]:
from sentence_transformers.losses import CoSENTLoss
loss = CoSENTLoss(model)

/var/folders/x2/vkkqdf2j73qc3q41102n2jt80000gn/T/ipykernel_33325/2256255756.py:1: DeprecationWarning: Importing from 'sentence_transformers.losses' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.losses' instead.
  from sentence_transformers.losses import CoSENTLoss


In [54]:
from sentence_transformers import SentenceTransformerTrainer
from sentence_transformers.training_args import SentenceTransformerTrainingArguments
from datasets import Dataset, DatasetDict, IterableDataset

/var/folders/x2/vkkqdf2j73qc3q41102n2jt80000gn/T/ipykernel_33325/969053275.py:2: DeprecationWarning: Importing from 'sentence_transformers.training_args' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.training_args' instead.
  from sentence_transformers.training_args import SentenceTransformerTrainingArguments


In [55]:
training_args = SentenceTransformerTrainingArguments(
    output_dir="./models/resume_job_matcher",
    
    num_train_epochs=1,
    
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    
    learning_rate=2e-5,
    
    warmup_ratio=0.1,
    
    eval_strategy="epoch",
    save_strategy="epoch",
    
    logging_steps=50,
    
    load_best_model_at_end=True,
    
    report_to="none"
)

In [56]:
print(train_dataset[0])

{'sentence1': 'SummaryI am seeking a position wherein I may use my proven development, technical, and analytical skills while having the opportunity to continue my learning and diversify my experiences. Highlights17 years experience in Information TechnologyHeld a Secret security clearanceExpert in business process including requirements gathering, analysis, design, development, testing and implementation of software applicationsStrong understanding of structured programming techniques and Software Development life Cycle (SDLC) - Agile (SCRUM)Able to learn, understand, and apply new software programs easily.Interface design and implementationAgile development methodologiesProject management and Requirements analysis and design phasesProcess improvementResults-orientedStaff developmentExcellent communication, analytical, interpersonal, and presentation skillsExpert ability at managing multiple projects simultaneously Experience06 2011to11 2015Manager of Software DevelopmentAvidxchange B

In [57]:
print(train_dataset)
print(val_dataset)

Dataset({
    features: ['sentence1', 'sentence2', 'score'],
    num_rows: 5099
})
Dataset({
    features: ['sentence1', 'sentence2', 'score'],
    num_rows: 1275
})


In [58]:
trainer = SentenceTransformerTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    loss=loss
)

In [59]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,3.305200,3.273944


TrainOutput(global_step=638, training_loss=3.545088920473679, metrics={'train_runtime': 2788.9941, 'train_samples_per_second': 1.828, 'train_steps_per_second': 0.229, 'total_flos': 0.0, 'train_loss': 3.545088920473679, 'epoch': 1.0})

In [60]:
fine_tuned_results = baseline_evaluator(model)

print(fine_tuned_results)

{'baseline_pearson_cosine': 0.48815316078684506, 'baseline_spearman_cosine': 0.46860468157954116}


In [61]:
baseline_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

baseline_results = baseline_evaluator(baseline_model)

print("BASELINE:")
print(baseline_results)

print("\nFINE-TUNED:")
print(fine_tuned_results)

BASELINE:
{'baseline_pearson_cosine': 0.2140447009238954, 'baseline_spearman_cosine': 0.20930305950683958}

FINE-TUNED:
{'baseline_pearson_cosine': 0.48815316078684506, 'baseline_spearman_cosine': 0.46860468157954116}


In [62]:
test_jobs = [
    {
        "title": "Machine Learning Engineer",
        "description": """
        We are looking for a Machine Learning Engineer with experience
        in Python, PyTorch, machine learning, deep learning, NLP,
        model development, and deploying ML systems using APIs.
        """
    },
    {
        "title": "Data Analyst",
        "description": """
        We are looking for a Data Analyst with strong Python, SQL,
        Excel, data visualization, dashboard development, statistics,
        and business analytics skills.
        """
    },
    {
        "title": "Frontend Developer",
        "description": """
        We are looking for a Frontend Developer with strong React,
        TypeScript, JavaScript, CSS, UI development, accessibility,
        and frontend application architecture experience.
        """
    },
    {
        "title": "Mining Engineer",
        "description": """
        We are looking for a Mining Engineer with experience in mine
        planning, geological modelling, mineral processing, drilling,
        blasting, mine safety, and mining operations.
        """
    }
]

In [63]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

results = []

candidate_embedding = model.encode(
    candidate_text,
    normalize_embeddings=True
)

for job in test_jobs:

    job_embedding = model.encode(
        job["description"],
        normalize_embeddings=True
    )

    score = cosine_similarity(
        [candidate_embedding],
        [job_embedding]
    )[0][0]

    results.append({
        "job": job["title"],
        "similarity": float(score)
    })

In [64]:
results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    "similarity",
    ascending=False
)

results_df

,job,similarity
0,Machine Learning Engineer,0.941322
1,Data Analyst,0.918967
3,Mining Engineer,0.894566
2,Frontend Developer,0.870728


In [65]:
model.save("./models/resume-job-matcher-v1")

In [66]:
def normalize_skills(skills):
    return {
        skill.strip().lower()
        for skill in skills
        if skill and isinstance(skill, str)
    }


candidate_skills = normalize_skills(
    candidate_profile["skills"]
)

print(candidate_skills)

{'neural networks', 'llm apis', 'rest apis', 'vector databases', 'regression', 'model evaluation', 'pydantic', 'langgraph', 'classification', 'vs code', 'docker', 'agents', 'retrieval-augmented generation (rag)', 'scikit-learn', 'numpy', 'feature engineering', 'javascript', 'css', 'git', 'matplotlib', 'hyperparameter tuning', 'jupyter notebook', 'prompt engineering', 'supervised learning', 'data preprocessing', 'tensorflow', 'html', 'streamlit', 'machine learning', 'ai', 'langchain', 'semantic search', 'python', 'keras', 'pandas', 'embeddings', 'deep learning', 'fastapi', 'transformers', 'github', 'xgboost', 'unsupervised learning', 'sql', 'pytorch'}


In [67]:
def calculate_skill_match(candidate_skills, job_skills):
    candidate = normalize_skills(candidate_skills)
    job = normalize_skills(job_skills)

    if not job:
        return 0.0

    matched = candidate.intersection(job)

    return len(matched) / len(job)

In [68]:
def normalize_job(
    job_id,
    title,
    company,
    description,
    location=None,
    employment_type=None,
    apply_url=None,
    source=None
):
    return {
        "job_id": job_id,
        "title": title,
        "company": company,
        "description": description,
        "location": location,
        "employment_type": employment_type,
        "apply_url": apply_url,
        "source": source
    }

In [69]:
def build_job_text(job):
    parts = [
        f"Job Title: {job.get('title', '')}",
        f"Company: {job.get('company', '')}",
        f"Location: {job.get('location', '')}",
        f"Employment Type: {job.get('employment_type', '')}",
        f"Description: {job.get('description', '')}"
    ]

    return "\n".join(parts)

In [70]:
def score_job(candidate_text, job, model):

    job_text = build_job_text(job)

    candidate_embedding = model.encode(
        candidate_text,
        normalize_embeddings=True
    )

    job_embedding = model.encode(
        job_text,
        normalize_embeddings=True
    )

    similarity = cosine_similarity(
        [candidate_embedding],
        [job_embedding]
    )[0][0]

    return float(similarity)

In [71]:
def rank_jobs(candidate_text, jobs, model):

    scored_jobs = []

    for job in jobs:
        score = score_job(
            candidate_text,
            job,
            model
        )

        job_copy = job.copy()
        job_copy["similarity_score"] = score

        scored_jobs.append(job_copy)

    return sorted(
        scored_jobs,
        key=lambda x: x["similarity_score"],
        reverse=True
    )

In [72]:
def recommendation_label(score):

    if score >= 0.85:
        return "HIGH PRIORITY"

    elif score >= 0.70:
        return "CONSIDER"

    else:
        return "LOW PRIORITY"

In [73]:
import requests
import xml.etree.ElementTree as ET
from bs4 import BeautifulSoup

In [74]:
def fetch_lever_jobs(company_slug):
    url = f"https://api.lever.co/v0/postings/{company_slug}?mode=xml"

    response = requests.get(url, timeout=30)
    response.raise_for_status()

    root = ET.fromstring(response.content)

    jobs = []

    for job in root.findall("job"):
        def get_text(tag):
            element = job.find(tag)
            return element.text.strip() if element is not None and element.text else ""

        jobs.append({
            "job_id": get_text("id"),
            "title": get_text("position"),
            "company": get_text("employer"),
            "description": get_text("description"),
            "location": get_text("location"),
            "category": get_text("category"),
            "employment_type": get_text("commitment"),
            "apply_url": get_text("apply_url"),
            "post_date": get_text("post_date"),
            "source": "Lever"
        })

    return jobs

In [75]:
level_ai_jobs = fetch_lever_jobs("levelai")

print("Number of jobs:", len(level_ai_jobs))

Number of jobs: 31


In [76]:
level_ai_jobs[:2]

[{'job_id': 'c55479cf-bc3e-471b-9ed7-ac213b0ee8b5',
  'title': 'AI Analyst',
  'company': 'Level AI',
  'description': '<div><span style="font-size: 11pt;">Level AI was founded in 2019 and is a Series C startup headquartered in Mountain View, California. Level AI revolutionises customer engagement by transforming contact centres into strategic assets. Our AI-native platform leverages advanced technologies such as Large Language Models to extract deep insights from customer interactions. By providing actionable intelligence, Level AI empowers organisations to enhance customer experience and drive growth. Consistently updated with the latest AI innovations, Level AI stands as the most adaptive and forward-thinking solution in the industry. As a critical member of the team, your work will be cutting-edge technologies and will play a high-impact role in shaping the future of AI-driven enterprise applications. You will directly work with people who\'ve worked at Amazon, Facebook, Google, an

In [77]:
def build_job_text(job):
    return f"""
Job Title: {job['title']}

Company: {job['company']}

Location: {job['location']}

Category: {job['category']}

Employment Type: {job['employment_type']}

Job Description:
{job['description']}
"""


In [78]:
matcher = SentenceTransformer(
    "./models/resume-job-matcher-v1"
)

In [79]:
candidate_embedding = matcher.encode(
    candidate_text,
    normalize_embeddings=True
)

job_texts = [
    build_job_text(job)
    for job in level_ai_jobs
]

job_embeddings = matcher.encode(
    job_texts,
    normalize_embeddings=True,
    show_progress_bar=True
)

similarities = cosine_similarity(
    [candidate_embedding],
    job_embeddings
)[0]

Batches: 100%|██████████| 1/1 [00:01<00:00,  1.59s/it]


In [80]:
for job, score in zip(level_ai_jobs, similarities):
    job["similarity_score"] = float(score)

In [81]:
ranked_jobs = sorted(
    level_ai_jobs,
    key=lambda x: x["similarity_score"],
    reverse=True
)

In [82]:
for i, job in enumerate(ranked_jobs[:10], start=1):

    print("=" * 70)

    print(f"{i}. {job['title']}")
    print(f"Company: {job['company']}")
    print(f"Location: {job['location']}")
    print(f"Type: {job['employment_type']}")
    print(f"Match: {job['similarity_score']:.3f}")
    print(f"Apply: {job['apply_url']}")

1. Senior Backend Engineer- AI Agents
Company: Level AI
Location: Noida
Type: Full Time
Match: 0.960
Apply: https://jobs.lever.co/levelai/98f0df69-79f8-4072-b676-19d9eec4df74/apply?lever-source=Job%20postings%20feed
2. Senior Backend Engineer- AI Agents
Company: Level AI
Location: Bangalore
Type: Full Time
Match: 0.960
Apply: https://jobs.lever.co/levelai/98f0df69-79f8-4072-b676-19d9eec4df74/apply?lever-source=Job%20postings%20feed
3. Senior Machine Learning Engineer (Bangalore/Noida Location)
Company: Level AI
Location: Bangalore
Type: Full Time
Match: 0.957
Apply: https://jobs.lever.co/levelai/55ba7aee-57ff-440c-91a4-58bcafcd10a8/apply?lever-source=Job%20postings%20feed
4. Senior Backend Engineer -PE
Company: Level AI
Location: Bengaluru
Type: Full Time
Match: 0.957
Apply: https://jobs.lever.co/levelai/195920b3-83c9-4f7d-8612-b6b7a252154d/apply?lever-source=Job%20postings%20feed
5. Senior Machine Learning Engineer (Bangalore/Noida Location)
Company: Level AI
Location: Noida
Type: Ful

In [83]:
jobs_df = pd.DataFrame(ranked_jobs)

jobs_df = jobs_df[
    [
        "title",
        "company",
        "location",
        "employment_type",
        "similarity_score",
        "apply_url",
        "source"
    ]
]

jobs_df.head(10)

,title,company,location,employment_type,similarity_score,apply_url,source
0,Senior Backend Engineer- AI Agents,Level AI,Noida,Full Time,0.960141,https://jobs.lever.co/levelai/98f0df69-79f8-40...,Lever
1,Senior Backend Engineer- AI Agents,Level AI,Bangalore,Full Time,0.959664,https://jobs.lever.co/levelai/98f0df69-79f8-40...,Lever
2,Senior Machine Learning Engineer (Bangalore/No...,Level AI,Bangalore,Full Time,0.957009,https://jobs.lever.co/levelai/55ba7aee-57ff-44...,Lever
3,Senior Backend Engineer -PE,Level AI,Bengaluru,Full Time,0.956899,https://jobs.lever.co/levelai/195920b3-83c9-4f...,Lever
4,Senior Machine Learning Engineer (Bangalore/No...,Level AI,Noida,Full Time,0.956676,https://jobs.lever.co/levelai/55ba7aee-57ff-44...,Lever
5,Senior Backend Engineer - Customer Engineering,Level AI,Bengaluru,Full Time,0.956654,https://jobs.lever.co/levelai/24d02a5c-3dc1-48...,Lever
6,Senior Backend Engineer - Customer Engineering,Level AI,Noida,Full Time,0.955427,https://jobs.lever.co/levelai/24d02a5c-3dc1-48...,Lever
7,Senior Backend Engineer -PE,Level AI,Noida,Full Time,0.955363,https://jobs.lever.co/levelai/195920b3-83c9-4f...,Lever
8,"Senior Site Reliability Engineer (Noida, BLR, ...",Level AI,Bengaluru,Full Time,0.955122,https://jobs.lever.co/levelai/0e0cd345-09e0-4d...,Lever
9,"Senior Full Stack Engineer (Noida, BLR, India)",Level AI,Bengaluru,Full Time,0.955056,https://jobs.lever.co/levelai/d3198817-960e-46...,Lever


In [84]:
def recommendation(score):

    if score >= 0.90:
        return "HIGH PRIORITY"

    elif score >= 0.80:
        return "CONSIDER"

    else:
        return "LOW PRIORITY"

In [85]:
jobs_df["recommendation"] = jobs_df[
    "similarity_score"
].apply(recommendation)

In [86]:
jobs_df = jobs_df.drop_duplicates(
    subset=["title", "company", "location"]
).reset_index(drop=True)

print("Jobs after deduplication:", len(jobs_df))

Jobs after deduplication: 31


In [88]:
target_roles = [
    "Data Analyst",
    "Data Scientist",
    "Machine Learning Engineer",
    "Machine Learning Engineer",
    "AI Engineer",
    "Business Analyst",
    "ML Engineer",
    "Junior Data Analyst"
]
candidate_experience_years = 1

In [89]:
def detect_seniority(title):

    title = title.lower()

    senior_keywords = [
        "senior",
        "sr.",
        "sr ",
        "lead",
        "principal",
        "staff",
        "director",
        "head",
        "manager"
    ]

    junior_keywords = [
        "junior",
        "jr.",
        "jr ",
        "entry level",
        "entry-level",
        "graduate",
        "intern",
        "trainee"
    ]

    if any(keyword in title for keyword in senior_keywords):
        return "senior"

    if any(keyword in title for keyword in junior_keywords):
        return "junior"

    return "mid"

In [90]:
for title in [
    "Senior Machine Learning Engineer",
    "Machine Learning Engineer",
    "Junior Data Analyst",
    "Data Analyst Intern"
]:
    print(title, "→", detect_seniority(title))

Senior Machine Learning Engineer → senior
Machine Learning Engineer → mid
Junior Data Analyst → junior
Data Analyst Intern → junior


In [110]:
def experience_match(title, job_description, candidate_experience_years):

    title_lower = title.lower()

    # Strong seniority mismatch
    if any(word in title_lower for word in [
        "senior",
        "sr.",
        "lead",
        "principal",
        "staff",
        "director",
        "head"
    ]):
        return 0.1

    # Management positions
    if any(word in title_lower for word in [
        "manager",
        "engineering manager"
    ]):
        return 0.1

    # Junior / entry-level positions
    if any(word in title_lower for word in [
        "junior",
        "jr.",
        "entry level",
        "entry-level",
        "graduate",
        "trainee",
        "intern"
    ]):
        return 1.0

    # Otherwise treat it as potentially suitable
    return 0.8

In [111]:
jobs_df["experience_match"] = jobs_df.apply(
    lambda row: experience_match(
        row["title"],
        row.get("description", ""),
        candidate_experience_years
    ),
    axis=1
)

In [112]:
jobs_df = pd.DataFrame(ranked_jobs)

In [113]:
jobs_df["experience_match"] = jobs_df.apply(
    lambda row: experience_match(
        row["title"],
        row["description"],
        candidate_experience_years
    ),
    axis=1
)

In [138]:
target_role_patterns = {
    "data analyst": [
        "data analyst",
        "junior data analyst",
        "senior data analyst",
        "business data analyst",
        "reporting analyst",
    ],

    "data scientist": [
        "data scientist",
        "junior data scientist",
    ],

    "machine learning engineer": [
        "machine learning engineer",
        "ml engineer",
    ],

    "ai engineer": [
        "ai engineer",
        "artificial intelligence engineer",
    ],

    "ai analyst": [
        "ai analyst",
        "artificial intelligence analyst",
    ],

    "business analyst": [
        "business analyst",
    ],

    "data analytics": [
        "data analytics",
        "analytics analyst",
        "analytics specialist",
    ],

    "nlp": [
        "nlp engineer",
        "nlp scientist",
        "natural language processing engineer",
    ],
}

In [139]:
def role_match(title):

    title = title.lower().strip()

    # Exact target-role matching
    for role, patterns in target_role_patterns.items():

        for pattern in patterns:

            if pattern in title:
                return 1.0

    # Related roles
    related_patterns = [
        "machine learning",
        "artificial intelligence",
        "data analytics",
        "analytics",
        "nlp",
        "data science",
    ]

    if any(pattern in title for pattern in related_patterns):
        return 0.6

    return 0.0

In [140]:
test_titles = [
    "AI Analyst",
    "Machine Learning Engineer",
    "Data Analyst",
    "SDET",
    "HR Generalist",
    "Revenue Operations Architect",
    "Forward Deployed Engineer - Agents"
]

for title in test_titles:
    print(title, "→", role_match(title))

AI Analyst → 1.0
Machine Learning Engineer → 1.0
Data Analyst → 1.0
SDET → 0.0
HR Generalist → 0.0
Revenue Operations Architect → 0.0
Forward Deployed Engineer - Agents → 0.0


In [141]:
jobs_df["role_match"] = jobs_df["title"].apply(
    role_match
)

In [142]:
jobs_df["final_score"] = (
    jobs_df["similarity_score"] * 0.60
    + jobs_df["role_match"] * 0.25
    + jobs_df["experience_match"] * 0.15
)

In [143]:
jobs_df[
    [
        "title",
        "company",
        "location",
        "similarity_score",
        "role_match",
        "experience_match",
        "final_score"
    ]
].head(20)

,title,company,location,similarity_score,role_match,experience_match,final_score
0,Sr. Machine Learning Engineer (Speech) - AI (B...,Level AI,Bengaluru,0.949514,1.0,0.1,0.834708
1,Sr. Machine Learning Engineer (Speech) - AI (B...,Level AI,Noida,0.949183,1.0,0.1,0.834510
2,Senior Machine Learning Engineer - NLP (Bangal...,Level AI,Bengaluru,0.954114,1.0,0.1,0.837468
3,Senior Machine Learning Engineer - NLP (Bangal...,Level AI,Noida,0.953438,1.0,0.1,0.837063
4,Senior Machine Learning Engineer (Bangalore/No...,Level AI,Bangalore,0.957009,1.0,0.1,0.839205
5,Senior Machine Learning Engineer (Bangalore/No...,Level AI,Noida,0.956676,1.0,0.1,0.839006
6,AI Analyst,Level AI,Noida,0.949790,1.0,0.8,0.939874
7,"Senior Backend Engineer - Analytics, Noida",Level AI,Bengaluru,0.952464,0.6,0.1,0.736478
8,"Senior Backend Engineer - Analytics, Noida",Level AI,Noida,0.950251,0.6,0.1,0.735151
9,"Senior Full Stack Engineer (Noida, BLR, India)",Level AI,Bengaluru,0.955056,0.0,0.1,0.588034


In [144]:
candidate_skills = {
    skill.lower().strip()
    for skill in candidate_profile["skills"]
    if isinstance(skill, str)
}

print(len(candidate_skills))
print(candidate_skills)

44
{'neural networks', 'llm apis', 'rest apis', 'vector databases', 'regression', 'model evaluation', 'pydantic', 'langgraph', 'classification', 'vs code', 'docker', 'agents', 'retrieval-augmented generation (rag)', 'scikit-learn', 'numpy', 'feature engineering', 'javascript', 'css', 'git', 'matplotlib', 'hyperparameter tuning', 'jupyter notebook', 'prompt engineering', 'supervised learning', 'data preprocessing', 'tensorflow', 'html', 'streamlit', 'machine learning', 'ai', 'langchain', 'semantic search', 'python', 'keras', 'pandas', 'embeddings', 'deep learning', 'fastapi', 'transformers', 'github', 'xgboost', 'unsupervised learning', 'sql', 'pytorch'}


In [145]:
skill_vocabulary = [
    "python",
    "sql",
    "machine learning",
    "deep learning",
    "pytorch",
    "tensorflow",
    "keras",
    "scikit-learn",
    "pandas",
    "numpy",
    "xgboost",
    "nlp",
    "transformers",
    "llm",
    "langchain",
    "langgraph",
    "rag",
    "fastapi",
    "docker",
    "git",
    "github",
    "javascript",
    "html",
    "css",
    "excel",
    "power bi",
    "tableau",
]

In [146]:
def extract_job_skills(job_description):

    text = job_description.lower()

    found = set()

    for skill in skill_vocabulary:

        if skill in text:
            found.add(skill)

    return found

In [147]:
def skill_match(job_description, candidate_skills):

    job_skills = extract_job_skills(job_description)

    if not job_skills:
        return 0.5

    candidate_skills_lower = {
        skill.lower().strip()
        for skill in candidate_skills
    }

    matched = job_skills.intersection(
        candidate_skills_lower
    )

    return len(matched) / len(job_skills)

In [148]:
jobs_df["skill_match"] = jobs_df.apply(
    lambda row: skill_match(
        row["description"],
        candidate_skills
    ),
    axis=1
)

In [149]:
jobs_df["final_score"] = (
    jobs_df["similarity_score"] * 0.50
    + jobs_df["skill_match"] * 0.25
    + jobs_df["role_match"] * 0.15
    + jobs_df["experience_match"] * 0.10
)

In [150]:
jobs_df = jobs_df.sort_values(
    "final_score",
    ascending=False
).reset_index(drop=True)

In [151]:
jobs_df[
    [
        "title",
        "company",
        "location",
        "similarity_score",
        "skill_match",
        "role_match",
        "experience_match",
        "final_score"
    ]
].head(20)

,title,company,location,similarity_score,skill_match,role_match,experience_match,final_score
0,AI Analyst,Level AI,Noida,0.949790,0.875000,1.0,0.8,0.923645
1,Sr. Machine Learning Engineer (Speech) - AI (B...,Level AI,Bengaluru,0.949514,0.800000,1.0,0.1,0.834757
2,Sr. Machine Learning Engineer (Speech) - AI (B...,Level AI,Noida,0.949183,0.800000,1.0,0.1,0.834591
3,Senior Machine Learning Engineer - NLP (Bangal...,Level AI,Bengaluru,0.954114,0.769231,1.0,0.1,0.829365
4,Senior Machine Learning Engineer - NLP (Bangal...,Level AI,Noida,0.953438,0.769231,1.0,0.1,0.829026
5,Senior Machine Learning Engineer (Bangalore/No...,Level AI,Bangalore,0.957009,0.625000,1.0,0.1,0.794754
6,Senior Machine Learning Engineer (Bangalore/No...,Level AI,Noida,0.956676,0.625000,1.0,0.1,0.794588
7,"Senior Backend Engineer - Analytics, Noida",Level AI,Bengaluru,0.952464,0.666667,0.6,0.1,0.742899
8,"Senior Backend Engineer - Analytics, Noida",Level AI,Noida,0.950251,0.666667,0.6,0.1,0.741792
9,"Senior Full Stack Engineer (Noida, BLR, India)",Level AI,Bengaluru,0.955056,0.857143,0.0,0.1,0.701814


In [230]:
# ============================================================
# SIMPLE JOB RECOMMENDATION
# ============================================================

# We already calculated:
# - similarity_score
# - skill_match
# - role_match
# - experience_match
# - final_score

# Don't throw jobs away at this stage.
# Just rank them.

recommended_jobs = (
    jobs_df
    .copy()
    .sort_values("final_score", ascending=False)
    .drop_duplicates(
        subset=["company", "title"]
    )
    .reset_index(drop=True)
)


# Keep the top 10 recommendations
recommended_jobs = recommended_jobs.head(10)


print("\n")
print("=" * 70)
print("TOP JOB RECOMMENDATIONS")
print("=" * 70)

print(
    recommended_jobs[
        [
            "title",
            "company",
            "location",
            "similarity_score",
            "skill_match",
            "role_match",
            "experience_match",
            "final_score"
        ]
    ].to_string(index=False)
)


print("\n")
print(
    f"Recommended jobs: {len(recommended_jobs)}"
)



TOP JOB RECOMMENDATIONS
                                                                  title  company             location  similarity_score  skill_match  role_match  experience_match  final_score
                                                             AI Analyst Level AI                Noida          0.949790     0.875000         1.0               0.8     0.923645
Sr. Machine Learning Engineer (Speech) - AI (Bangalore/Noida Locations) Level AI            Bengaluru          0.949514     0.800000         1.0               0.1     0.834757
      Senior Machine Learning Engineer - NLP (Bangalore/Noida Location) Level AI            Bengaluru          0.954114     0.769231         1.0               0.1     0.829365
            Senior Machine Learning Engineer (Bangalore/Noida Location) Level AI            Bangalore          0.957009     0.625000         1.0               0.1     0.794754
                             Senior Backend Engineer - Analytics, Noida Level AI            Be

In [241]:
# Keep all jobs for ranking.
# Eligibility is retained as diagnostic information,
# but it does not remove potentially useful recommendations.

eligible_jobs = jobs_df.copy()

In [242]:
eligible_jobs = eligible_jobs.sort_values(
    "final_score",
    ascending=False
).reset_index(drop=True)

In [243]:
eligible_jobs[
    [
        "title",
        "company",
        "location",
        "similarity_score",
        "skill_match",
        "role_match",
        "experience_match",
        "final_score"
    ]
].head(20)

,title,company,location,similarity_score,skill_match,role_match,experience_match,final_score
0,AI Analyst,Level AI,Noida,0.949790,0.875000,1.0,0.8,0.923645
1,Sr. Machine Learning Engineer (Speech) - AI (B...,Level AI,Bengaluru,0.949514,0.800000,1.0,0.1,0.834757
2,Sr. Machine Learning Engineer (Speech) - AI (B...,Level AI,Noida,0.949183,0.800000,1.0,0.1,0.834591
3,Senior Machine Learning Engineer - NLP (Bangal...,Level AI,Bengaluru,0.954114,0.769231,1.0,0.1,0.829365
4,Senior Machine Learning Engineer - NLP (Bangal...,Level AI,Noida,0.953438,0.769231,1.0,0.1,0.829026
5,Senior Machine Learning Engineer (Bangalore/No...,Level AI,Bangalore,0.957009,0.625000,1.0,0.1,0.794754
6,Senior Machine Learning Engineer (Bangalore/No...,Level AI,Noida,0.956676,0.625000,1.0,0.1,0.794588
7,"Senior Backend Engineer - Analytics, Noida",Level AI,Bengaluru,0.952464,0.666667,0.6,0.1,0.742899
8,"Senior Backend Engineer - Analytics, Noida",Level AI,Noida,0.950251,0.666667,0.6,0.1,0.741792
9,"Senior Full Stack Engineer (Noida, BLR, India)",Level AI,Bengaluru,0.955056,0.857143,0.0,0.1,0.701814


In [244]:
eligible_jobs["job_group"] = (
    eligible_jobs["company"].fillna("").str.lower().str.strip()
    + " | "
    + eligible_jobs["title"].fillna("").str.lower().str.strip()
)

In [245]:
recommended_jobs[
    [
        "title",
        "company",
        "location",
        "similarity_score",
        "skill_match",
        "role_match",
        "experience_match",
        "final_score",
        "apply_url"
    ]
]

,title,company,location,similarity_score,skill_match,role_match,experience_match,final_score,apply_url
0,AI Analyst,Level AI,Noida,0.949790,0.875000,1.0,0.8,0.923645,https://jobs.lever.co/levelai/c55479cf-bc3e-47...
1,Sr. Machine Learning Engineer (Speech) - AI (B...,Level AI,Bengaluru,0.949514,0.800000,1.0,0.1,0.834757,https://jobs.lever.co/levelai/78ddcdbd-4bd8-41...
2,Senior Machine Learning Engineer - NLP (Bangal...,Level AI,Bengaluru,0.954114,0.769231,1.0,0.1,0.829365,https://jobs.lever.co/levelai/cc04ab77-6ee3-40...
3,Senior Machine Learning Engineer (Bangalore/No...,Level AI,Bangalore,0.957009,0.625000,1.0,0.1,0.794754,https://jobs.lever.co/levelai/55ba7aee-57ff-44...
4,"Senior Backend Engineer - Analytics, Noida",Level AI,Bengaluru,0.952464,0.666667,0.6,0.1,0.742899,https://jobs.lever.co/levelai/6d4ce92d-88d9-4d...
5,"Senior Full Stack Engineer (Noida, BLR, India)",Level AI,Bengaluru,0.955056,0.857143,0.0,0.1,0.701814,https://jobs.lever.co/levelai/d3198817-960e-46...
6,"SDET (Noida, India)",Level AI,Noida,0.945032,0.500000,0.0,0.8,0.677516,https://jobs.lever.co/levelai/55678283-3e23-46...
7,EA and Operations Specialist - Bay Area,Level AI,"Bay Area, California",0.919468,0.500000,0.0,0.8,0.664734,https://jobs.lever.co/levelai/5f299aa0-d358-43...
8,Senior Frontend Engineer,Level AI,Noida,0.949377,0.600000,0.0,0.1,0.634689,https://jobs.lever.co/levelai/e6658081-92af-4f...
9,Senior Backend Engineer -PE,Level AI,Bengaluru,0.956899,0.571429,0.0,0.1,0.631306,https://jobs.lever.co/levelai/195920b3-83c9-4f...


In [246]:
def recommendation_label(score):
    if score >= 0.80:
        return "HIGH PRIORITY"
    elif score >= 0.70:
        return "GOOD MATCH"
    elif score >= 0.60:
        return "CONSIDER"
    else:
        return "LOW MATCH"

In [247]:
recommended_jobs["recommendation"] = (
    recommended_jobs["final_score"]
    .apply(recommendation_label)
)

print("Recommended jobs:", len(recommended_jobs))

Recommended jobs: 10


In [248]:
jobs_df[
    ~jobs_df["eligible"]
][
    [
        "title",
        "role_match",
        "eligibility_reason"
    ]
].head(20)

,title,role_match,eligibility_reason
1,Sr. Machine Learning Engineer (Speech) - AI (B...,1.0,Seniority mismatch
2,Sr. Machine Learning Engineer (Speech) - AI (B...,1.0,Seniority mismatch
3,Senior Machine Learning Engineer - NLP (Bangal...,1.0,Seniority mismatch
4,Senior Machine Learning Engineer - NLP (Bangal...,1.0,Seniority mismatch
5,Senior Machine Learning Engineer (Bangalore/No...,1.0,Seniority mismatch
6,Senior Machine Learning Engineer (Bangalore/No...,1.0,Seniority mismatch
7,"Senior Backend Engineer - Analytics, Noida",0.6,Seniority mismatch
8,"Senior Backend Engineer - Analytics, Noida",0.6,Seniority mismatch
9,"Senior Full Stack Engineer (Noida, BLR, India)",0.0,Seniority mismatch
10,"Senior Full Stack Engineer (Noida, BLR, India)",0.0,Seniority mismatch


In [249]:
recommended_jobs[
    [
        "title",
        "company",
        "location",
        "similarity_score",
        "skill_match",
        "role_match",
        "experience_match",
        "final_score",
        "recommendation",
        "apply_url"
    ]
]

,title,company,location,similarity_score,skill_match,role_match,experience_match,final_score,recommendation,apply_url
0,AI Analyst,Level AI,Noida,0.949790,0.875000,1.0,0.8,0.923645,HIGH PRIORITY,https://jobs.lever.co/levelai/c55479cf-bc3e-47...
1,Sr. Machine Learning Engineer (Speech) - AI (B...,Level AI,Bengaluru,0.949514,0.800000,1.0,0.1,0.834757,HIGH PRIORITY,https://jobs.lever.co/levelai/78ddcdbd-4bd8-41...
2,Senior Machine Learning Engineer - NLP (Bangal...,Level AI,Bengaluru,0.954114,0.769231,1.0,0.1,0.829365,HIGH PRIORITY,https://jobs.lever.co/levelai/cc04ab77-6ee3-40...
3,Senior Machine Learning Engineer (Bangalore/No...,Level AI,Bangalore,0.957009,0.625000,1.0,0.1,0.794754,GOOD MATCH,https://jobs.lever.co/levelai/55ba7aee-57ff-44...
4,"Senior Backend Engineer - Analytics, Noida",Level AI,Bengaluru,0.952464,0.666667,0.6,0.1,0.742899,GOOD MATCH,https://jobs.lever.co/levelai/6d4ce92d-88d9-4d...
5,"Senior Full Stack Engineer (Noida, BLR, India)",Level AI,Bengaluru,0.955056,0.857143,0.0,0.1,0.701814,GOOD MATCH,https://jobs.lever.co/levelai/d3198817-960e-46...
6,"SDET (Noida, India)",Level AI,Noida,0.945032,0.500000,0.0,0.8,0.677516,CONSIDER,https://jobs.lever.co/levelai/55678283-3e23-46...
7,EA and Operations Specialist - Bay Area,Level AI,"Bay Area, California",0.919468,0.500000,0.0,0.8,0.664734,CONSIDER,https://jobs.lever.co/levelai/5f299aa0-d358-43...
8,Senior Frontend Engineer,Level AI,Noida,0.949377,0.600000,0.0,0.1,0.634689,CONSIDER,https://jobs.lever.co/levelai/e6658081-92af-4f...
9,Senior Backend Engineer -PE,Level AI,Bengaluru,0.956899,0.571429,0.0,0.1,0.631306,CONSIDER,https://jobs.lever.co/levelai/195920b3-83c9-4f...


In [250]:
recommended_jobs.to_csv(
    "../data/recommended_jobs.csv",
    index=False
)

In [251]:
print("Saved:", len(recommended_jobs), "recommended jobs")

Saved: 10 recommended jobs


In [252]:
# ============================================================
# CREATE APPLICATION QUEUE
# ============================================================

application_queue = recommended_jobs[
    [
        "job_id",
        "title",
        "company",
        "location",
        "apply_url",
        "final_score",
        "recommendation"
    ]
].copy()

application_queue = application_queue.rename(
    columns={
        "final_score": "match_score"
    }
)

application_queue["status"] = "PENDING"

application_queue.to_csv(
    "../data/application_queue.csv",
    index=False
)

print("Application queue created.")
print("Jobs in queue:", len(application_queue))

application_queue[
    [
        "job_id",
        "title",
        "company",
        "match_score",
        "recommendation",
        "status"
    ]
]

Application queue created.
Jobs in queue: 10


,job_id,title,company,match_score,recommendation,status
0,c55479cf-bc3e-471b-9ed7-ac213b0ee8b5,AI Analyst,Level AI,0.923645,HIGH PRIORITY,PENDING
1,78ddcdbd-4bd8-4187-b7d0-38fcb39c8232,Sr. Machine Learning Engineer (Speech) - AI (B...,Level AI,0.834757,HIGH PRIORITY,PENDING
2,cc04ab77-6ee3-4078-9cfd-110cda0b1438,Senior Machine Learning Engineer - NLP (Bangal...,Level AI,0.829365,HIGH PRIORITY,PENDING
3,55ba7aee-57ff-440c-91a4-58bcafcd10a8,Senior Machine Learning Engineer (Bangalore/No...,Level AI,0.794754,GOOD MATCH,PENDING
4,6d4ce92d-88d9-4d78-b30d-ae22b12a268f,"Senior Backend Engineer - Analytics, Noida",Level AI,0.742899,GOOD MATCH,PENDING
5,d3198817-960e-4625-9b06-418ca2e6ce4e,"Senior Full Stack Engineer (Noida, BLR, India)",Level AI,0.701814,GOOD MATCH,PENDING
6,55678283-3e23-46f0-baaa-48b91b5ff6ea,"SDET (Noida, India)",Level AI,0.677516,CONSIDER,PENDING
7,5f299aa0-d358-43ae-a2a4-58b9ac1d426d,EA and Operations Specialist - Bay Area,Level AI,0.664734,CONSIDER,PENDING
8,e6658081-92af-4f0c-aec8-c63ec7ded03e,Senior Frontend Engineer,Level AI,0.634689,CONSIDER,PENDING
9,195920b3-83c9-4f7d-8612-b6b7a252154d,Senior Backend Engineer -PE,Level AI,0.631306,CONSIDER,PENDING


In [253]:
application_queue = pd.read_csv(
    "../data/application_queue.csv"
)

print("Jobs in application queue:", len(application_queue))

application_queue

Jobs in application queue: 10


,job_id,title,company,location,apply_url,match_score,recommendation,status
0,c55479cf-bc3e-471b-9ed7-ac213b0ee8b5,AI Analyst,Level AI,Noida,https://jobs.lever.co/levelai/c55479cf-bc3e-47...,0.923645,HIGH PRIORITY,PENDING
1,78ddcdbd-4bd8-4187-b7d0-38fcb39c8232,Sr. Machine Learning Engineer (Speech) - AI (B...,Level AI,Bengaluru,https://jobs.lever.co/levelai/78ddcdbd-4bd8-41...,0.834757,HIGH PRIORITY,PENDING
2,cc04ab77-6ee3-4078-9cfd-110cda0b1438,Senior Machine Learning Engineer - NLP (Bangal...,Level AI,Bengaluru,https://jobs.lever.co/levelai/cc04ab77-6ee3-40...,0.829365,HIGH PRIORITY,PENDING
3,55ba7aee-57ff-440c-91a4-58bcafcd10a8,Senior Machine Learning Engineer (Bangalore/No...,Level AI,Bangalore,https://jobs.lever.co/levelai/55ba7aee-57ff-44...,0.794754,GOOD MATCH,PENDING
4,6d4ce92d-88d9-4d78-b30d-ae22b12a268f,"Senior Backend Engineer - Analytics, Noida",Level AI,Bengaluru,https://jobs.lever.co/levelai/6d4ce92d-88d9-4d...,0.742899,GOOD MATCH,PENDING
5,d3198817-960e-4625-9b06-418ca2e6ce4e,"Senior Full Stack Engineer (Noida, BLR, India)",Level AI,Bengaluru,https://jobs.lever.co/levelai/d3198817-960e-46...,0.701814,GOOD MATCH,PENDING
6,55678283-3e23-46f0-baaa-48b91b5ff6ea,"SDET (Noida, India)",Level AI,Noida,https://jobs.lever.co/levelai/55678283-3e23-46...,0.677516,CONSIDER,PENDING
7,5f299aa0-d358-43ae-a2a4-58b9ac1d426d,EA and Operations Specialist - Bay Area,Level AI,"Bay Area, California",https://jobs.lever.co/levelai/5f299aa0-d358-43...,0.664734,CONSIDER,PENDING
8,e6658081-92af-4f0c-aec8-c63ec7ded03e,Senior Frontend Engineer,Level AI,Noida,https://jobs.lever.co/levelai/e6658081-92af-4f...,0.634689,CONSIDER,PENDING
9,195920b3-83c9-4f7d-8612-b6b7a252154d,Senior Backend Engineer -PE,Level AI,Bengaluru,https://jobs.lever.co/levelai/195920b3-83c9-4f...,0.631306,CONSIDER,PENDING


In [254]:
from urllib.parse import urlparse


def extract_lever_posting_id(apply_url):
    path_parts = urlparse(
        apply_url
    ).path.strip("/").split("/")

    if len(path_parts) >= 2:
        return path_parts[1]

    return None

In [255]:
approved_jobs = application_queue[
    application_queue["status"] == "USER_APPROVED"
]

print("Approved applications:", len(approved_jobs))

if approved_jobs.empty:
    print("No USER_APPROVED applications yet.")

Approved applications: 0
No USER_APPROVED applications yet.


In [256]:
if not approved_jobs.empty:
    approved_job = approved_jobs.iloc[0]
    print(approved_job["apply_url"])
else:
    approved_job = None
    print("Approve a job first before extracting its application URL.")

Approve a job first before extracting its application URL.


In [257]:
if approved_job is not None:
    posting_id = extract_lever_posting_id(
        approved_job["apply_url"]
    )
    print("Posting ID:", posting_id)
else:
    posting_id = None

In [258]:
if approved_job is not None:
    print(approved_job["apply_url"])
    print(posting_id)
else:
    print("No approved job selected.")

No approved job selected.


In [259]:
application_queue = pd.read_csv(
    "../data/application_queue.csv"
)

print("Jobs in queue:", len(application_queue))

print(
    application_queue[
        [
            "job_id",
            "title",
            "company",
            "location",
            "match_score",
            "recommendation",
            "status"
        ]
    ].to_string(index=False)
)

Jobs in queue: 10
                              job_id                                                                   title  company             location  match_score recommendation  status
c55479cf-bc3e-471b-9ed7-ac213b0ee8b5                                                              AI Analyst Level AI                Noida     0.923645  HIGH PRIORITY PENDING
78ddcdbd-4bd8-4187-b7d0-38fcb39c8232 Sr. Machine Learning Engineer (Speech) - AI (Bangalore/Noida Locations) Level AI            Bengaluru     0.834757  HIGH PRIORITY PENDING
cc04ab77-6ee3-4078-9cfd-110cda0b1438       Senior Machine Learning Engineer - NLP (Bangalore/Noida Location) Level AI            Bengaluru     0.829365  HIGH PRIORITY PENDING
55ba7aee-57ff-440c-91a4-58bcafcd10a8             Senior Machine Learning Engineer (Bangalore/Noida Location) Level AI            Bangalore     0.794754     GOOD MATCH PENDING
6d4ce92d-88d9-4d78-b30d-ae22b12a268f                              Senior Backend Engineer - Analytics, Noid

In [260]:
# ============================================================
# APPROVE ONE JOB FOR TESTING
# ============================================================

application_queue = pd.read_csv(
    "../data/application_queue.csv"
)

# Approve the highest-ranked job
application_queue.loc[
    application_queue["match_score"].idxmax(),
    "status"
] = "USER_APPROVED"

application_queue.to_csv(
    "../data/application_queue.csv",
    index=False
)

print(
    application_queue[
        [
            "job_id",
            "title",
            "match_score",
            "recommendation",
            "status"
        ]
    ].head(10).to_string(index=False)
)

                              job_id                                                                   title  match_score recommendation        status
c55479cf-bc3e-471b-9ed7-ac213b0ee8b5                                                              AI Analyst     0.923645  HIGH PRIORITY USER_APPROVED
78ddcdbd-4bd8-4187-b7d0-38fcb39c8232 Sr. Machine Learning Engineer (Speech) - AI (Bangalore/Noida Locations)     0.834757  HIGH PRIORITY       PENDING
cc04ab77-6ee3-4078-9cfd-110cda0b1438       Senior Machine Learning Engineer - NLP (Bangalore/Noida Location)     0.829365  HIGH PRIORITY       PENDING
55ba7aee-57ff-440c-91a4-58bcafcd10a8             Senior Machine Learning Engineer (Bangalore/Noida Location)     0.794754     GOOD MATCH       PENDING
6d4ce92d-88d9-4d78-b30d-ae22b12a268f                              Senior Backend Engineer - Analytics, Noida     0.742899     GOOD MATCH       PENDING
d3198817-960e-4625-9b06-418ca2e6ce4e                          Senior Full Stack Engineer (Noid

In [262]:
import pandas as pd

queue_path = "../data/application_queue.csv"

application_queue = pd.read_csv(queue_path)

job_id = "c55479cf-bc3e-471b-9ed7-ac213b0ee8b5"

application_queue.loc[
    application_queue["job_id"] == job_id,
    "status"
] = "USER_APPROVED"

application_queue.to_csv(
    queue_path,
    index=False
)

print(
    application_queue[
        [
            "job_id",
            "title",
            "status"
        ]
    ].to_string(index=False)
)

                              job_id                                                                   title        status
c55479cf-bc3e-471b-9ed7-ac213b0ee8b5                                                              AI Analyst USER_APPROVED
78ddcdbd-4bd8-4187-b7d0-38fcb39c8232 Sr. Machine Learning Engineer (Speech) - AI (Bangalore/Noida Locations)       PENDING
cc04ab77-6ee3-4078-9cfd-110cda0b1438       Senior Machine Learning Engineer - NLP (Bangalore/Noida Location)       PENDING
55ba7aee-57ff-440c-91a4-58bcafcd10a8             Senior Machine Learning Engineer (Bangalore/Noida Location)       PENDING
6d4ce92d-88d9-4d78-b30d-ae22b12a268f                              Senior Backend Engineer - Analytics, Noida       PENDING
d3198817-960e-4625-9b06-418ca2e6ce4e                          Senior Full Stack Engineer (Noida, BLR, India)       PENDING
55678283-3e23-46f0-baaa-48b91b5ff6ea                                                     SDET (Noida, India)       PENDING
5f299aa0-d358-43